# LangDMTA Eval - Evaluating Test Questions Using LLM Judges

This notebook demonstrates how to use the `langdmta_eval` package to evaluate drug discovery agentic system outputs.

## What This Package Does

The `langdmta_eval` package provides:
- **Judge Signatures**: Pre-configured DSPy judges for evaluating agent outputs
- **Score Normalization**: Automatic conversion from categorical to numeric scores
- **Tool Validation**: Checking tool call sequences against expected patterns
- **Async Batch Evaluation**: Concurrent evaluation with progress tracking


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

from langdmta_eval import (
    load_results_from_json,
    validate_tool_sequence_batch,
    plot_score_histograms,
    plot_score_by_category,
    plot_score_heatmap,
    results_to_dataframe,
    plot_metric_correlation,
    plot_radar_chart,
)
from langdmta_eval.visualization.visualize import (
    plot_lineplot_variation,
    plot_score_pie
    )

print("✅ Visualization functions imported successfully")

## Loading test cases from CSV (Real Data)

From Langfuse CSV exports.

In [ ]:
results_path = "../../results/judge_results/test-questions-260429_trial_1_LLM_evaluation_results_comprehensive_v2_openai_google_gemini-3.1-pro-preview_optimized.json"save_path = "results/judge_results/test-questions-260429_trial_1_LLM_evaluation_results_comprehensive_v2_openai_google_gemini-3.1-pro-preview_optimized_filtered.json"

csv_results = load_results_from_json(results_path)

# Renaming tool from "Molformer" to more well known "Mol2Mol"
for results in csv_results:
    if results.test_case.test_name == "Molformer":
        results.test_case.test_name = "Mol2Mol"

scope_adherence_remap = {0.5: 1, 0:0, 1:0}  # Mapping 0.5 (ON TARGET) to 1 and 0 (BELOW TARGET) and 1 (ABOVE TARGET) to 0 "

for result in csv_results:
    raw = result.scores["scope_adherence"]
    result.scores["scope_adherence"] = scope_adherence_remap[raw]

# Specify path to folder save the figures
SAVE_DIR = "/evaluate_judge_figures/round2/"


In [ ]:
# Keep only relevant metrics — edit this list to control which metrics appear in all plots
RELEVANT_METRICS = ['completeness', 'relevancy', 'structural_clarity', 'scope_adherence', 'tool_call_correctness']
print(f"Relevant metrics: {RELEVANT_METRICS}")

# Tool Validation in Batch

Validate tool call sequences against expected patterns using batch processing with concurrency control.

In [ ]:

test_cases_from_csv = []
for result in csv_results:
    test_cases_from_csv.append(result.test_case)


# Validate all test cases in batch
validation_results = await validate_tool_sequence_batch(
    test_cases=test_cases_from_csv,
    max_concurrent=10,
    verbose=True
)

# Display results
print("\n" + "="*80)
print("BATCH VALIDATION RESULTS")
print("="*80)

for i, result in enumerate(validation_results):
    test_case = result['test_case']
    print(f"\nTest {i+1}: {test_case.test_name}")
    print(f"  Question: {test_case.question[:80]}...")
    print(f"  Tool calls: {test_case.tool_calls}")
    print(f"  Correctness: {result['correctness']}")
    print(f"  Is correct: {result['is_complete']}")

# Summary statistics
correct_count = sum(1 for r in validation_results if r['is_complete'])
complete_count = sum(1 for r in validation_results if r['correctness'] == 'complete')
partial_count = sum(1 for r in validation_results if r['correctness'] == 'partial')
incorrect_count = sum(1 for r in validation_results if r['correctness'] == 'incorrect')

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total test cases: {len(validation_results)}")
print(f"Correct (complete or partial): {correct_count} ({correct_count/len(validation_results)*100:.1f}%)")
print(f"  - Complete: {complete_count} ({complete_count/len(validation_results)*100:.1f}%)")
print(f"  - Partial: {partial_count} ({partial_count/len(validation_results)*100:.1f}%)")
print(f"Incorrect: {incorrect_count} ({incorrect_count/len(validation_results)*100:.1f}%)")

In [ ]:
# Validate all test cases in batch
validation_results = await validate_tool_sequence_batch(
    test_cases=test_cases_from_csv,
    max_concurrent=10,  # Adjust based on your needs
    verbose=True
)

# Get summary statistics
correct_count = sum(1 for r in validation_results if r['is_complete'])
partial_count = sum(1 for r in validation_results if r['is_partial'])
accuracy = correct_count / len(validation_results) * 100
partial_accuracy = partial_count / len(validation_results) * 100

print(f"Complete Count: {correct_count}")
print(f"Partial Count: {partial_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Partial Accuracy: {partial_accuracy:.2f}%")

In [ ]:
# adding tool_call_correctness metric as a score
TOOL_CORRECTNESS_MAP = {"complete": 1.0, "partial": 0.5}

for result, validation in zip(csv_results, validation_results):
    correctness = validation["correctness"]
    result.scores["tool_call_correctness"] = TOOL_CORRECTNESS_MAP.get(correctness, 0.0)

print(f"Injected 'tool_call_correctness' into {len(csv_results)} results")

## Visualizing Evaluation Results

Visualizations to analyze LLM judge evaluation.

**Grouping Options**

The grouping visualizations support the following grouping options:
- `category`: Group by test category
- `test_name`: Group by test name/type
- `session`: Group by session identifier
- `question_variation`: Group by question variation (if available in your data)

### Distributions of scores per category

In [ ]:
# 1. Plot score histograms for all metrics
fig = plot_score_histograms(csv_results, metrics=RELEVANT_METRICS + ["overall"], figsize=(20, 10), bins=3)

print("\nHistograms show the distribution of scores across all evaluation metrics.")

In [ ]:

scores = plot_score_pie(csv_results, metrics=RELEVANT_METRICS + ["overall"], figsize=(10, 8),
                        save_path=SAVE_DIR + "score_distribution_pie_chart.pdf"
                        )

### Summary of scores

In [ ]:
# 4. Convert to DataFrame for custom analysis
df = results_to_dataframe(csv_results)

print("Evaluation Results DataFrame:")
print(df[['category', 'test_name'] + RELEVANT_METRICS].head())

print("\n📊 Summary Statistics:")
print(df[RELEVANT_METRICS].describe())

### OPTIONAL: Remove samples with errors: BadRequestError, PolicyError, GraphRecursionError etc.

In [ ]:
if False: # Change to True to filter out cases with errors in agent output and re-plot visualizations
    print(len(csv_results))
    types_of_errors = ["BadRequestError", "policy error", "JSON error", "GraphRecursionError", "recursion limit", "content filter", "unhandled exception"]

    filtered_csv_results = []
    for result in csv_results:
        no_error = True
        for error in types_of_errors:
            if error in result.test_case.agent_output:
                no_error = False
        if no_error:
            filtered_csv_results.append(result)

    # assing csv_results to filtered_csv_results for visualizations
    csv_results = filtered_csv_results
    print(len(csv_results))    

### Scores by Categories

In [ ]:
# 1. Plot scores grouped by category: workflow vs Tool
fig = plot_score_by_category(
    csv_results,
    group_by='category',
    metrics=RELEVANT_METRICS + ["overall"],
    figsize=(10, 5),
    save_path=SAVE_DIR + "scores_by_category.pdf",
    show_mean=True
)

print("\nBar plot shows average scores across different question categories.")

In [ ]:
# 2. Plot scores grouped by category: question type (test_name)
fig = plot_score_by_category(
    csv_results,
    group_by='test_name',
    metrics=RELEVANT_METRICS + ["overall"],
    figsize=(16, 8),
    show_mean=False,

)

print("\nBar plot shows average scores across different question categories.")

In [ ]:
# 3a. Plot scores grouped by category: question variation (formality levels)
fig = plot_score_by_category(
    csv_results,
    group_by='question_variation',
    metrics=RELEVANT_METRICS + ["overall"],
    figsize=(10, 5),
    save_path=SAVE_DIR + "scores_by_question_variation.pdf",
    show_mean=False
)

print("\nBar plot shows average scores across different question categories.")

In [ ]:
# 3b. Plot scores grouped by category: question variation (formality levels)
fig = plot_lineplot_variation(
    csv_results,
    group_by='question_variation',
    metrics=RELEVANT_METRICS + ["overall"],
    figsize=(16, 8)
)

print("\nLine plot shows trends in overall scores across different question variations.")

## Plot score heatmap to show distribution of scores per category

In [ ]:
# 3. Plot heatmap for correctness scores by category
for metric in RELEVANT_METRICS + ["overall"]:
    fig = plot_score_heatmap(
        csv_results,
        metric=metric,
        group_by='test_name',
        pool_metrics=RELEVANT_METRICS,
        figsize=(10, 8)
    )

print("\nHeatmap shows the frequency distribution of correctness scores (0.0, 0.5, 1.0) across categories.")

In [ ]:

metrics = RELEVANT_METRICS + ["overall"]
print(f"Metrics for visualizations: {metrics}")

### Radar plots for multi-dimensional comparison across groups

In [ ]:
# 4. Radar Chart - Multi-dimensional comparison across groups
fig = plot_radar_chart(csv_results, group_by='test_name', metrics=metrics, figsize=(14, 6)
                       , save_path=SAVE_DIR + "radar_chart_by_test_name.pdf"
                       )

print("\nRadar chart shows multi-metric performance comparison across different test types.")

In [ ]:
# 4. Radar Chart - Multi-dimensional comparison across groups
fig = plot_radar_chart(csv_results, group_by='question_variation', metrics=metrics, figsize=(12, 6))

print("\nRadar chart shows multi-metric performance comparison across different test types.")

### Correlation Matrix - showing how the metrics correlate to each other

In [ ]:
fig = plot_metric_correlation(csv_results, metrics=metrics, figsize=(12, 10), 
                              save_path=SAVE_DIR + "metric_correlation_heatmap.pdf"
                              )


### Print individual samples

In [ ]:
count = 0

for i in range(len(csv_results)):
    if csv_results[i].scores["scope_adherence"] == 0:
        count += 1
        print("="*40, i, "="*40)
        print("Question: ", csv_results[i].test_case.question)
        print("Agent Output: ", csv_results[i].test_case.agent_output)
        print("Expected Tool Calls: ", csv_results[i].test_case.expected_tool_calls)
        print("Actual Tool Calls: ", csv_results[i].test_case.tool_calls)
        print("Accepted")
        print("Reasoning: ", csv_results[i].reasoning)
        print("Scores: ", csv_results[i].scores)
print("\n")
print(f"Total cases: {count} out of 70")